Rerunning the initial criteria cuts, to see if I accidentally lost some galaxies

In [1]:
from astropy.table import Table, vstack
import numpy as np

from tqdm import tqdm
from astropy import units as u
from astropy.constants import c

import sys
sys.path.append('..')
from galaxy_selection import *

from velocity_map_fxns import *

import os

import matplotlib.pyplot as plt
import matplotlib as mpl

21443 galaxies

In [2]:
loa_1 = Table.read('/pscratch/sd/d/dbustos/rot_curves/loa_targs_v2.fits')
loa_missing = Table.read('/pscratch/sd/d/dbustos/rot_curves/loa_targs_missing.fits')
# loa_table = Table.read('/pscratch/sd/d/dbustos/rot_curves/loa_targs_v2.fits')

loa_table = vstack((loa_1, loa_missing), join_type = 'outer')

loa_table[:5]

TARGETID,SGA_ID,TARGET_RA,TARGET_DEC,Z,Z_ERR,ZWARN,DELTACHI2,DIST,DIST_R26,PA,C_TO_F_ANGLE,ANGLE_OFF_AXIS,Selection,ZERR_MOD,unique_obs,Velocity,V_err,Z_center
int64,int64,float64,float64,float64,float64,int64,float64,float64,float64,float64,float64,float64,int64,float64,float64,float64,float64,float64
1084427169955849,556089,242.72318746350186,53.635782093163655,0.03063192091855999,0.03063192091855999,0,789.0584922824055,0.0017610627660455075,0.32999999905624167,44.0717658996582,224.0717662682332,0.0,1,--,--,--,--,--
1084434300272644,375898,243.65642926326169,54.27724177905015,0.07850942434688926,0.07850942434688926,0,310.9083716990426,0.0018719092656299759,0.32999999823339154,54.93254852294922,54.93254783937784,0.0,1,--,--,--,--,--
1084427178344451,21266,243.62537013307835,53.712014537285526,0.05723634298805001,0.05723634298805001,0,849.3460895419121,0.0015670324751866896,0.32999999454262413,111.70044708251953,111.70044970679623,0.0,1,--,--,--,--,--
1084434300272643,375898,243.65118121249577,54.275090804072406,0.07974131508417623,0.07974131508417623,0,1123.4490712519037,0.001871909265610506,0.3299999982299592,54.93254852294922,234.93254783934097,0.0,1,--,--,--,--,--
1084427178344450,21266,243.62044993111272,53.713173370492754,0.05591772316967483,0.05591772316967483,0,2463.962895900011,0.0015670324751904594,0.329999994543418,111.70044708251953,291.700449706759,0.0,1,--,--,--,--,--


In [3]:
SGA = Table.read('/global/cfs/cdirs/cosmo/data/sga/2020/SGA-2020.fits', 'ELLIPSE')

SGA_dict = {}
for i in range(len(SGA)):
    SGA_dict[SGA['SGA_ID'][i]] = i

First, get the deprojected distances to remove points outside R26

In [4]:
def deproj_dist(proj_dist, angle, ba):
    q0 = .2

    rho = np.radians(angle)

    cos2i = (ba**2 - q0**2)/(1 - q0**2)

    x = np.cos(angle)

    y = np.sin(angle)

    deprojected = dist * np.sqrt((x**2) + (y**2/cos2i))

    return(deprojected)

In [5]:
# get deprojected distance
loa_table['deproj_dist'] = np.nan
loa_table['deproj_r26'] = np.nan


for i in tqdm(np.unique(loa_table['SGA_ID'])):
    
    sga_idx = SGA_dict[i]

    targ_id = loa_table['SGA_ID'] == i
    
    #targs = shredded_new[shredded_new['SGA_ID'] == i]

    dist = loa_table['DIST'][targ_id]

    phi = loa_table['ANGLE_OFF_AXIS'][targ_id]

    axis_ratio = SGA['BA'][sga_idx]

    r26_dist = .5 * SGA['D26'][sga_idx]

    dep = deproj_dist(dist, phi, axis_ratio)

    dep_r26 = dep*u.deg.to('arcmin')/r26_dist

    # deprojected distance in units degrees
    loa_table['deproj_dist'][targ_id] = dep

    # deprojected distance in r26
    loa_table['deproj_r26'][targ_id] = dep_r26

  0%|          | 0/21540 [00:00<?, ?it/s]/tmp/ipykernel_914506/1084790706.py:12: RuntimeWarning: invalid value encountered in sqrt
  deprojected = dist * np.sqrt((x**2) + (y**2/cos2i))
100%|██████████| 21540/21540 [00:06<00:00, 3385.16it/s]


In [6]:
# only take rows with deprojected r26 < 1.1 
loa_table = loa_table[loa_table['deproj_r26'] < 1.1]

First, run a code to see how many unique observations are in each galaxy
Previously, I just used dist_r26, but I'm going to change it to deproj_r26

In [11]:
loa_table['unique_obs'] = np.nan

for i in tqdm(np.unique(loa_table['SGA_ID'])):
    
    fiber = loa_table['SGA_ID'] == i 

    sga_id = SGA_dict[i] 

    obs = loa_table[fiber]

    # identify all points on semimajor axis
    on_major = obs[obs['ANGLE_OFF_AXIS'] == 0]
    if len(on_major) == 0:
        unique_on = 0

    else:
        if (SGA['PA'][sga_id] < 45) or (SGA['PA'][sga_id] > 135):

            left_index = on_major['TARGET_DEC'] - SGA['DEC'][sga_id] > 0

        else:
            left_index = on_major['TARGET_RA'] - SGA['RA'][sga_id] > 0
                    
        left = on_major[left_index]
        right = on_major[~left_index]
        
        left['deproj_r26'] = np.round(left['deproj_r26'], 6)
        right['deproj_r26'] = np.round(right['deproj_r26'], 6)
    
        left_on_major = left.group_by('deproj_r26')
        right_on_major = right.group_by('deproj_r26')

        left_len = len(left_on_major.groups) if len(left) > 0 else 0 
        right_len = len(right_on_major.groups) if len(right) > 0 else 0 

        unique_on = left_len + right_len

    # identify all points off major axis
    off_major = obs[obs['ANGLE_OFF_AXIS'] != 0]
    
    # find number of unique points off axis
    num_off_major = len(off_major)

    if num_off_major == 0:
        unique_off = 0
    else:
        off_major = off_major.group_by(['deproj_r26','ANGLE_OFF_AXIS'])
        unique_off = len(off_major.groups)

    total_obs = unique_on + unique_off

    loa_table['unique_obs'][fiber] = total_obs

100%|██████████| 21528/21528 [03:56<00:00, 90.97it/s] 


Now, redo a criteria cut

In [12]:
center_dist_lim = 0.001
unique_dist_lim = 0.01

In [13]:
# redo criteria cut
#empty column to classify galaxy
loa_table['Selection'] = 0

#for each unique SGA ID
for i in tqdm(np.unique(loa_table['SGA_ID'])):

    #identify all the galaxies and targets
    obs_id = loa_table['SGA_ID'] == i
    
    #makes a table of the targets corresponding to this galaxy
    obs = loa_table[obs_id]

    sga_id = SGA_dict[i] 

    criteria_a = False
    criteria_b = False

#if the galaxy has more than three observations
    if len(obs) >= 3:

#-----
# a
#-----
        #check to see if there is a center observation
        if np.any(obs['deproj_r26'] <= center_dist_lim):
            # test_counter += 1
        
            # check how many unique points on semimajor axis
            not_center = obs[obs['deproj_r26'] > unique_dist_lim]

            if len(not_center) >= 2 and (np.max(not_center['deproj_r26']) - np.min(not_center['deproj_r26'])) >= unique_dist_lim:
                
                #since there 2 unique points, classify it as viable galaxy
                loa_table['Selection'][obs_id] = 1
                criteria_a = True

# #----
# # b
# #----
        elif len(obs) >= 10:
            distances = sorted(obs['deproj_r26'])
            counter = 0

            for i in range(len(distances)-1):
                if distances[i+1] - distances[i] > unique_dist_lim:
                    counter += 1
            
            if counter >= 10:
                loa_table['Selection'][obs_id] = 2
                criteria_b = True

        
#----
# c
#----
          
        elif not (criteria_a or criteria_b):
            
            # identify all points on semimajor axis
            on_major = obs[obs['ANGLE_OFF_AXIS'] == 0]

            # identify all points off major axis
            off_major = obs[obs['ANGLE_OFF_AXIS'] != 0]


            if len(on_major) > 0:
            
                #split targets on semimajor axis onto either side of center galaxy
                if (SGA['PA'][sga_id] < 45) or (SGA['PA'][sga_id] > 135):

                    left_index = on_major['TARGET_DEC'] - SGA['DEC'][sga_id] > 0

                else:
                    left_index = on_major['TARGET_RA'] - SGA['RA'][sga_id] > 0
                
                left = on_major[left_index]
                right = on_major[~left_index]

            if len(left) > 0 and len(right) > 0:

                for j in range(len(left)):
                    
                    # check that there are 2 symmetric observations
                    if np.any(np.abs(right['deproj_r26'] - left['deproj_r26'][j]) < unique_dist_lim):

                    #check if there is a third point
                        if (np.any(np.abs(right['deproj_r26'] - left['deproj_r26'][j]) > unique_dist_lim) 
                            or np.any(np.abs(left['deproj_r26'] - left['deproj_r26'][j]) > unique_dist_lim)
                            or np.any(np.abs(off_major['deproj_r26'] - left['deproj_r26'][j]) > unique_dist_lim)):

                            #viable galaxy
                            loa_table['Selection'][obs_id] = 3
                            
    # if tf_loa['Selection'][obs_id][0] == 1:
    #     print(obs['SGA_ID'][0])

print(len(np.unique(loa_table['SGA_ID'][loa_table['Selection'] != 0])))

100%|██████████| 21528/21528 [00:39<00:00, 538.23it/s]

10432


In [14]:
loa_table = loa_table[loa_table['Selection'] != 0]

In [15]:
len(np.unique(loa_table['SGA_ID']))

10432

This brought # galaxies from 21443 to 10336. Now run a velocity cut, without deprojecting the velocities. Previously, when I ran this cut, I didn't remove points outside R26. I was left with ~11000 galaxies

In [16]:
# new error correcting for redrock 7km/s systematic uncertainty
dv_sys = 7 #km/s
dz_sys = dv_sys/c.to('km/s').value
loa_table['ZERR_MOD'] = np.sqrt(loa_table['Z_ERR']**2 + (dz_sys*(1 + loa_table['Z']))**2)

In [17]:
zwarn = 0
deltachi2 = 25

center_dist_lim = 0.001
unique_dist_lim = 0.01

q0 = 0.2

In [31]:
loa_table['Velocity'] = np.nan
loa_table['V_err'] = np.nan
loa_table['Z_center'] = np.nan
#loa['c_or_s'] = 0

c = c.to('km/s')

for i in tqdm(np.unique(loa_table['SGA_ID'])):

    has_valid_center = False
    
    fiber = loa_table['SGA_ID'] == i

    obs = loa_table[fiber]

    #find the index for this target in SGA
    #sga_idx = SGA_dict[i]
    
    # find z and its error for all targets
    z_targ = loa_table['Z'][fiber]
    z_e = loa_table['ZERR_MOD'][fiber]

    # identify all points on semimajor axis
    on_major = obs[obs['ANGLE_OFF_AXIS'] == 0]

    off_major = obs[obs['ANGLE_OFF_AXIS'] != 0]

    total_obs = obs['unique_obs'][0]

#-----------------------------------------------------------------------
# find center redshift if there is a center target that meets criteria
#-----------------------------------------------------------------------
    
    # check for center fiber
    if np.any(on_major['deproj_r26'] < center_dist_lim):
        
        # find all center fibers that fit quality check
        criteria_c = (on_major['deproj_r26'] < center_dist_lim)
        center = on_major[criteria_c & criteria_sym(on_major, zwarn, deltachi2)]

        # find z for center targets
        z_c = center['Z']

        # calculate weight of center targets
        zc_err = center['ZERR_MOD']
        weight = 1/(zc_err ** 2)
        #print(weight)
                
        # get weighted average
        # if weight = 0, then it did not meet criteria. 
        if np.any(weight > 0):
            has_valid_center = True
            z_cen = np.average(z_c, weights = weight)
            z_cen_err = np.sqrt(np.sum(zc_err ** 2))
    
#--------------------------------------------------------------
# find the center redshift given two symmetric points
#--------------------------------------------------------------
    if not has_valid_center:

        sga_id = SGA_dict[i]

        #split targets on semimajor axis onto either side of center galaxy
        if (SGA['PA'][sga_id] < 45) or (SGA['PA'][sga_id] > 135):

            left_index = on_major['TARGET_DEC'] - SGA['DEC'][sga_id] > 0

        else:
            left_index = on_major['TARGET_RA'] - SGA['RA'][sga_id] > 0
            
        left = on_major[left_index]
        right = on_major[~left_index]

        #----------------------------------------------------------
        # identify where the symmetric points are and group them ------------
        #----------------------------------------------------------
        #find R26 for each fiber
        right_dist, left_dist = np.array(right['deproj_r26']), np.array(left['deproj_r26'])

        #create a matrix subtracting each right r26 element from each left r26 element
        diff_matrix = np.abs(right_dist[:,np.newaxis]-left_dist)

        #identify all points in matrix where difference is within unique dist (fibers are symmetric)
        right_idx, left_idx = np.where(diff_matrix < unique_dist_lim)

        #make sure there are symmetric points
        if (len(right_idx) == 0) or (len(left_idx) == 0):  
            
            # if not, check if there are 10 unique points
            if total_obs >= 10:
                # get the average redshift
                weight = 1/(z_e **2)
                z_cen = np.average(z_targ, weights = weight)
                z_cen_err = np.sqrt(np.sum(z_e**2))
                
            else:
                # print(sga_id, 'no center, symmetric point, or random pts')
                continue  
        
        elif (len(right_idx) != 0) and (len(left_idx) != 0):
    
            #put those fibers into appropriate tables
            right['deproj_r26'] = np.round(right['deproj_r26'],6)
            left['deproj_r26'] = np.round(left['deproj_r26'],6)
            
            symmetric_right = right[np.unique(right_idx)].group_by('deproj_r26')
            symmetric_left = left[np.unique(left_idx)].group_by('deproj_r26')
    
            # print('left: ', symmetric_left['DIST_R26'], len(symmetric_left.groups))
            # print('right:', symmetric_right['DIST_R26'], len(symmetric_right.groups))
            
    #--------------------------------------
    # get pseudo-center for each grouping
    #--------------------------------------
            # number of fiber groups
            len_sym = len(symmetric_right.groups)
      
            #empty array for z_c and weight for each fiber group
            z_c = np.empty(len_sym)
            weight = np.empty(len_sym)
            
        #-----------------------   
        # for each fiber group
        #-----------------------
            for z in range(len_sym):
                
                #-----------------------------------------
                # get right points
                #-----------------------------------------
                right_group = symmetric_right.groups[z]
                
                # check if there is more than one observation
                if len(right_group) > 1:
                    # if there is, only take the good observations
                    if np.any(right_group['ZWARN'] == 0) and np.any(right_group['DELTACHI2'] > 25):
                        right_group = right_group[criteria_sym(right_group, zwarn, deltachi2)]
                
                #-----------------------------------------
                # get left points
                #-----------------------------------------
                left_group = symmetric_left.groups[z]
                
                # check if there is more than one observation
                if len(left_group) > 1:
                    # if there is, only take the good observations
                    if np.any(left_group['ZWARN'] == 0) and np.any(left_group['DELTACHI2'] > 25):
                        left_group = left_group[criteria_sym(left_group, zwarn, deltachi2)]
                        
                #--------------------------------
                # get redshift and pseudo center
                #--------------------------------
                # get average z for right and left
                z_right = right_group['Z'].mean()
                z_left = left_group['Z'].mean()
    
                # pseudo z_center for fiber group
                z_c[z] = (z_right+z_left)/2
    
                # propagate uncertainty
                zr_err = np.sum(right_group['ZERR_MOD'] ** 2)
                zl_err = np.sum(left_group['ZERR_MOD'] ** 2)
                
                z_err = np.sqrt(zr_err + zl_err)/2
              
                # weight for each fiber group
                weight[z] = 1/(z_err ** 2)
    
        #----------------------------------------------------
        # if multiple symmetric groups, remove the bad pairs
        #----------------------------------------------------
            if len_sym > 1:
                
                # copy array of z_c and weight
                zc_pairs = z_c.copy()
                weight_pairs = weight.copy()
                
                for pair in range(len_sym):
                    right_group = symmetric_right.groups[pair]
                    right_group = right_group[criteria_sym(right_group, zwarn, deltachi2)]
                
                    left_group = symmetric_left.groups[pair]
                    left_group = left_group[criteria_sym(left_group, zwarn, deltachi2)]
    
                    # if all fibers in the group are bad, then remove it from redshift and weight calculation
                    
                    if (len(right_group) == 0) or (len(left_group) == 0):
                        zc_pairs[pair] = 0
                        weight_pairs[pair] = 0
                        
                # if every pair is bad, then keep all the pairs
                if np.sum(zc_pairs) == 0:
                    # print(i,'all bad pairs zc')
                    z_c = z_c
                else:
                    z_c = zc_pairs
    
                if np.sum(weight_pairs) == 0:
                    # print(i,'all bad pair weight')
                    weight = weight
                else:
                    weight = weight_pairs 
    
        #-----------------------------------------------------
        # calculate weighted pseudo z_center and error
        #-----------------------------------------------------
            z_cen = np.average(z_c, weights = weight)
            
            z_cen_err = np.sqrt(np.sum(z_err**2))
    
            # print('pseudo:',z_cen)
    
#---------------------------------------------------------
#find the redshift of each fiber relative to the center
#---------------------------------------------------------

    # relative redshift
    z_rel = (1 + z_targ)/(1 + z_cen) - 1

    #inclination angle
    # axis_ratio = SGA['BA'][sga_idx]
    # inc = sin_i(axis_ratio, q0)
    
    # find the rotational velocity
    velocity = z_rel*c

    # rotational velocity error
    v_error = c*np.sqrt((z_cen_err**2)+(z_e**2))
    
    counter = np.where(abs(velocity.value) < 1000, 1, 0)

    if np.sum(counter) < 3:
        #print (sga_idx, ':not enough points')
        loa_table['Velocity'][fiber] = np.nan
        loa_table['V_err'][fiber] = np.nan

    elif np.sum(counter) >= 3:
        loa_table['Velocity'][fiber] = velocity  

        loa_table['V_err'][fiber] = v_error
    
        loa_table['Z_center'][fiber] = z_cen_err



        
        # find number unique observations

        
    #     valid_mask = (loa_table['Velocity'][fiber] > -1000 ) & (loa_table['Velocity'][fiber] < 1000 )
    #     on_major_filtered = on_major[valid_mask[obs['ANGLE_OFF_AXIS']==0]]
       
    #     if len(on_major_filtered) == 0:
    #         unique_on = 0

    #     else:
    #         on_major_filtered['deproj_r26'] = np.round(on_major_filtered['deproj_r26'], 6)
    #         on_major_filtered = on_major_filtered.group_by('deproj_r26')
    #         unique_on_filtered = len(on_major_filtered.groups)

    #     # identify all points off major axis
    #     off_major_filtered = off_major[valid_mask[obs['ANGLE_OFF_AXIS']!=0]]
    
    #     # find number of unique points off axis
    #     num_off_major = len(off_major_filtered)

    #     if num_off_major == 0:
    #         unique_off = 0
    #     else:
    #         off_major_filtered = off_major_filtered.group_by(['deproj_r26','ANGLE_OFF_AXIS'])
    #         unique_off = len(off_major_filtered.groups)
    
    #     total_obs = unique_on + unique_off

    #     if total_obs < 3:
    #         print(loa_table[fiber]['Velocity'])
    #         print(loa_table[fiber]['dist_r26'])
    #         # loa_table['Velocity'][fiber] = np.nan
    #         # loa_table['V_err'][fiber] = np.nan
    #     else:
    #         # print(loa_table[fiber]['Velocity'])
    #         continue

    #         # loa_table['Velocity'][fiber] = velocity  

    #         # loa_table['V_err'][fiber] = v_error
    
    #         # loa_table['Z_center'][fiber] = z_cen_err
        
    # #print(velocity)

 42%|████▏     | 4380/10432 [00:15<00:20, 301.34it/s]/tmp/ipykernel_914506/884245566.py:154: RuntimeWarning: Mean of empty slice
  z_left = left_group['Z'].mean()
/global/common/software/nersc/pe/conda-envs/26.8.1/python-3.13/nersc-python/lib/python3.13/site-packages/numpy/_core/_methods.py:142: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
100%|██████████| 10432/10432 [00:35<00:00, 293.95it/s]


In [32]:
# create a new table that removes all NAN
loa_table = loa_table[np.isfinite(loa_table['Velocity'])]
print(len(np.unique(tf_loa['SGA_ID'])))

6806


This brought the # of galaxies before VI from 10880 to 6809 (-4071 galaxies)

In [39]:
loa_table['unique_obs_vel'] = np.nan

In [42]:
# remove galaxies that have < 3 unique observations that are < 1000 km/s

for i in tqdm(np.unique(loa_table['SGA_ID'])):
    
    fiber = loa_table['SGA_ID'] == i 

    sga_id = SGA_dict[i] 

    points = loa_table[fiber]

    vel = points['Velocity']
        
    obs = points[(vel > -1000) & (vel < 1000)]


    # identify all points on semimajor axis
    on_major = obs[obs['ANGLE_OFF_AXIS'] == 0]
    if len(on_major) == 0:
        unique_on = 0

    else:
        if (SGA['PA'][sga_id] < 45) or (SGA['PA'][sga_id] > 135):

            left_index = on_major['TARGET_DEC'] - SGA['DEC'][sga_id] > 0

        else:
            left_index = on_major['TARGET_RA'] - SGA['RA'][sga_id] > 0
                    
        left = on_major[left_index]
        right = on_major[~left_index]
        
        left['deproj_r26'] = np.round(left['deproj_r26'], 6)
        right['deproj_r26'] = np.round(right['deproj_r26'], 6)
    
        left_on_major = left.group_by('deproj_r26')
        right_on_major = right.group_by('deproj_r26')

        left_len = len(left_on_major.groups) if len(left) > 0 else 0 
        right_len = len(right_on_major.groups) if len(right) > 0 else 0 

        unique_on = left_len + right_len

    # identify all points off major axis
    off_major = obs[obs['ANGLE_OFF_AXIS'] != 0]
    
    # find number of unique points off axis
    num_off_major = len(off_major)

    if num_off_major == 0:
        unique_off = 0
    else:
        off_major = off_major.group_by(['deproj_r26','ANGLE_OFF_AXIS'])
        unique_off = len(off_major.groups)

    total_obs = unique_on + unique_off

    loa_table['unique_obs_vel'][fiber] = total_obs

100%|██████████| 6806/6806 [01:10<00:00, 96.69it/s] 


In [45]:
tf_loa = loa_table[loa_table['unique_obs_vel'] >= 3]
print(len(np.unique(tf_loa['SGA_ID'])))

5997


In [48]:
tf_loa[tf_loa['SGA_ID']==126182]

TARGETID,SGA_ID,TARGET_RA,TARGET_DEC,Z,Z_ERR,ZWARN,DELTACHI2,DIST,DIST_R26,PA,C_TO_F_ANGLE,ANGLE_OFF_AXIS,Selection,ZERR_MOD,unique_obs,Velocity,V_err,Z_center,deproj_dist,deproj_r26,unique_obs_vel
int64,int64,float64,float64,float64,float64,int64,float64,float64,float64,float64,float64,float64,int64,float64,float64,float64,float64,float64,float64,float64,float64


In [49]:
tf_loa.write('/pscratch/sd/d/dbustos/Fall_26/loa_before_VI.fits',format='fits',overwrite=True)